# Dr. P.E.A.D.S. — Full Engine Launcher
**Paediatric Education and Diagnostic Support**

Run each cell in order:
1. Install dependencies
2. Clone the engine files from GitHub
3. Set your API keys
4. **Engine 1** — Confirm database presence
5. **Engine 2** — Link & test the P.A.E.D.S. API
6. **Engine 3** — Launch Dr. P.E.A.D.S. chatbot

> Upload your question bank JSON files to `question_banks/` before running Engine 1.

---
## Cell 1 — Install dependencies

In [ ]:
!pip install -q chromadb sentence-transformers requests google-generativeai
# Uncomment the line for your preferred LLM:
# !pip install -q openai
# !pip install -q anthropic
print('Dependencies installed.')

---
## Cell 2 — Download engine files from GitHub

In [ ]:
import os, urllib.request

BRANCH = 'claude/chatbot-build-instructions-GIJQn'
REPO   = 'R-DAWG-dev/Atomic-curriculum-embedder'
BASE   = f'https://raw.githubusercontent.com/{REPO}/{BRANCH}'

files = [
    'engine_1_db_check.py',
    'engine_2_api_link.py',
    'engine_3_dr_peads_chatbot.py',
]

for f in files:
    urllib.request.urlretrieve(f'{BASE}/{f}', f)
    print(f'Downloaded: {f}')

os.makedirs('question_banks', exist_ok=True)
print('\nAll engine files ready. Upload your question banks to question_banks/ now.')

---
## Cell 3 — Set API keys
Add secrets in the **Colab Secrets panel** (key icon in the left sidebar):

| Secret name | Value |
|---|---|
| `PAEDS_API_KEY` | Your P.A.E.D.S. API key (ask the admin) |
| `GOOGLE_API_KEY` | Your Gemini API key (or `OPENAI_API_KEY` / `ANTHROPIC_API_KEY`) |

Toggle **Notebook access ON** for each secret, then run this cell.

In [ ]:
from google.colab import userdata

for secret in ('PAEDS_API_KEY', 'GOOGLE_API_KEY'):
    try:
        val = userdata.get(secret)
        if val:
            import os; os.environ[secret] = val
            print(f'[OK]  {secret} loaded ({len(val)} chars).')
        else:
            print(f'[WARN] {secret} not found in Colab Secrets.')
    except Exception as e:
        print(f'[WARN] Could not load {secret}: {e}')

---
## Cell 4 — Engine 1: Confirm database presence

In [ ]:
from engine_1_db_check import check_database, load_question_bank
import json, glob

# Auto-load any JSON question banks found in question_banks/
bank_files = glob.glob('question_banks/*.json')
if bank_files:
    all_pairs = []
    for path in bank_files:
        with open(path) as f:
            data = json.load(f)
        # Accept either a list of {question, answer} dicts or a dict with that list
        if isinstance(data, list):
            all_pairs.extend(data)
        elif isinstance(data, dict) and 'qa_pairs' in data:
            all_pairs.extend(data['qa_pairs'])
    print(f'Found {len(all_pairs)} Q&A pairs across {len(bank_files)} file(s). Loading...')
    load_question_bank(all_pairs, reset=False)
else:
    print('[INFO] No JSON files in question_banks/ yet. Skipping load.')

# Health check
result = check_database()
print(f'\nDatabase status: {result["status"]} | Records: {result["record_count"]}')

---
## Cell 5 — Engine 2: Link & test the P.A.E.D.S. API

In [ ]:
from engine_2_api_link import get_api_key, health_check, ask_paeds

paeds_key = get_api_key()
api_ok    = health_check(api_key=paeds_key)

if api_ok:
    test = ask_paeds('What is the normal respiratory rate for a newborn?', api_key=paeds_key)
    print(f'\nTest answer: {test}')
else:
    print('\n[WARN] API not reachable — Engine 3 will rely solely on the local database.')

---
## Cell 6 — Engine 3: Launch Dr. P.E.A.D.S.
Change `provider` to `'openai'` or `'claude'` if you prefer a different LLM.

In [ ]:
from engine_3_dr_peads_chatbot import build_bot, run_chat

bot = build_bot('gemini')   # 'gemini' | 'openai' | 'claude'
run_chat(bot, provider_label='Gemini')

---
## Single-question mode (no chat loop)
Useful for testing without an interactive prompt.

In [ ]:
from engine_3_dr_peads_chatbot import build_bot

bot   = build_bot('gemini')
reply = bot.chat('What is the normal heart rate for a toddler?')
print(f'Dr. P.E.A.D.S.: {reply}')